# 7.1 Conectando Neo4j + Qdrant (LOCAL)

Integración con **Qdrant Docker** y **Neo4j Desktop** para búsqueda semántica de Recalls, Investigations y Complaints.

## ⚠️ Requisitos:

1. **Qdrant Docker** corriendo en `http://localhost:6333`
2. **Neo4j Desktop** corriendo en `neo4j://127.0.0.1:7687`
3. **Embeddings subidos** a Qdrant local (notebook 6.1)
4. **Datos subidos** a Neo4j (notebooks 4.1.1, 4.2.1, 4.3)

In [1]:
!pip install -q qdrant-client neo4j sentence-transformers torch pyvis jinja2

In [6]:
import os

# Neo4j Desktop LOCAL
os.environ["NEO4J_URI"] = "neo4j://127.0.0.1:7687"
os.environ["NEO4J_USER"] = "neo4j"
os.environ["NEO4J_PASS"] = "proyectotec"  # Cambiar a tu password

# Qdrant Docker LOCAL
os.environ["QDRANT_URL"] = "http://localhost:6333"
os.environ["QDRANT_API_KEY"] = ""  # No necesario en local

# Modelo E5
os.environ["E5_MODEL_NAME"] = "intfloat/multilingual-e5-large-instruct"
os.environ["E5_MAX_LEN"] = "256"

print("✅ Configuración LOCAL lista")
print(f"Neo4j: {os.environ['NEO4J_URI']}")
print(f"Qdrant: {os.environ['QDRANT_URL']}")

✅ Configuración LOCAL lista
Neo4j: neo4j://127.0.0.1:7687
Qdrant: http://localhost:6333


In [7]:
import os
import numpy as np
from neo4j import GraphDatabase
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
import torch

print("✅ Librerías importadas")

c:\Users\moral\anaconda3\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


✅ Librerías importadas


In [8]:
def get_env():
    env = {
        "NEO4J_URI": os.getenv("NEO4J_URI", ""),
        "NEO4J_USER": os.getenv("NEO4J_USER", ""),
        "NEO4J_PASSWORD": os.getenv("NEO4J_PASS", ""),
        "QDRANT_URL": os.getenv("QDRANT_URL", ""),
        "QDRANT_API_KEY": os.getenv("QDRANT_API_KEY", ""),
        "E5_MODEL_NAME": os.getenv("E5_MODEL_NAME", "intfloat/multilingual-e5-large-instruct"),
        "E5_MAX_LEN": int(os.getenv("E5_MAX_LEN", "256")),
    }
    assert env["NEO4J_URI"] and env["NEO4J_USER"] and env["NEO4J_PASSWORD"], "Config Neo4j incompleta."
    assert env["QDRANT_URL"], "Config Qdrant incompleta (QDRANT_URL)."
    return env

ENV = get_env()
print("✅ Configuración validada")

✅ Configuración validada


In [9]:
# ---------- Clientes perezosos ----------
_neo_driver = None
_qdrant_client = None
_e5_encoder = None

def get_neo_driver():
    global _neo_driver
    if _neo_driver is None:
        _neo_driver = GraphDatabase.driver(
            ENV["NEO4J_URI"],
            auth=(ENV["NEO4J_USER"], ENV["NEO4J_PASSWORD"])
        )
        _neo_driver.verify_connectivity()
    return _neo_driver

def get_qdrant_client():
    global _qdrant_client
    if _qdrant_client is None:
        _qdrant_client = QdrantClient(
            url=ENV["QDRANT_URL"],
            api_key=ENV["QDRANT_API_KEY"] if ENV["QDRANT_API_KEY"] else None,
            timeout=180
        )
    return _qdrant_client

def get_encoder():
    global _e5_encoder
    if _e5_encoder is not None:
        return _e5_encoder

    model_name = ENV["E5_MODEL_NAME"]
    max_len    = ENV["E5_MAX_LEN"]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    _e5_encoder = SentenceTransformer(model_name, device=device)
    _e5_encoder.max_seq_length = max_len
    return _e5_encoder

print("✅ Clientes inicializados")

✅ Clientes inicializados


In [10]:
@torch.no_grad()
def e5_query(text: str) -> np.ndarray:
    """Genera embedding de consulta usando E5"""
    enc = get_encoder()
    vec = enc.encode(
        [f"query: {text}"],
        normalize_embeddings=True,
        show_progress_bar=False
    )
    return np.asarray(vec[0], dtype=np.float32)

print("✅ Función e5_query lista")

✅ Función e5_query lista


## Función ask() - Búsqueda Semántica + Enriquecimiento Neo4j

In [11]:
def ask(query_text: str, k: int = 10, type_in: str = "recall"):
    """
    Busca semánticamente en Qdrant y enriquece con Neo4j.
    
    Args:
        query_text: Query en lenguaje natural
        k: Número de resultados
        type_in: Tipo de colección ('recall', 'investigation', 'complaint')
    """
    # 1. Generar embedding
    query_vec = e5_query(query_text)
    
    # 2. Buscar en Qdrant
    collection_name = f"nhtsa_{type_in}s"
    client = get_qdrant_client()
    
    results = client.search(
        collection_name=collection_name,
        query_vector=query_vec.tolist(),
        limit=k,
        with_payload=True
    )
    
    # 3. Extraer camp_no de los payloads (no el id numérico)
    recalled_ids = []
    hits_data = []
    
    for hit in results:
        # El payload tiene 'id' que es el camp_no real
        recall_id = hit.payload.get('id') or hit.payload.get('camp_no', '')
        if recall_id and recall_id != 'NONE':
            recalled_ids.append(recall_id)
            hits_data.append({
                'id': recall_id,
                'score': hit.score,
                'payload': hit.payload
            })
    
    print(f"🔍 Encontrados {len(hits_data)} resultados en Qdrant")
    
    # 4. Consultar Neo4j con los camp_no reales
    driver = get_neo_driver()
    enriched = []
    
    with driver.session() as s:
        for data in hits_data:
            recall_id = data['id']
            
            # Complaints no tienen relaciones con Components, solo propiedad 'component'
            if type_in == 'complaint':
                query = f"""
                MATCH (r:{type_in.capitalize()} {{id: $id}})
                OPTIONAL MATCH (r)-[:OF_MAKE]->(mk:Make)
                OPTIONAL MATCH (r)-[:OF_MODEL]->(md:Model)
                RETURN r, mk.name AS make, md.name AS model, 
                       [r.component] AS components
                """
            else:
                query = f"""
                MATCH (r:{type_in.capitalize()} {{id: $id}})
                OPTIONAL MATCH (r)-[:OF_MAKE]->(mk:Make)
                OPTIONAL MATCH (r)-[:OF_MODEL]->(md:Model)
                OPTIONAL MATCH (r)-[:MENTIONS]->(comp:Component)
                RETURN r, mk.name AS make, md.name AS model, 
                       collect(DISTINCT comp.name) AS components
                """
            result = s.run(query, id=recall_id).single()
            
            if result:
                # Obtener texto completo del payload
                full_text = data['payload'].get('text', '') or ''
                
                # Para diferentes tipos de entidades, usar campos apropiados
                if type_in == 'investigation':
                    # Investigations tienen 'summary' y 'subject'
                    subject = result['r'].get('subject', '')
                    summary = result['r'].get('summary', '')
                    full_text = full_text or summary or subject
                    date_field = result['r'].get('open_date', '')
                elif type_in == 'complaint':
                    # Complaints tienen 'description'
                    description = result['r'].get('description', '')
                    full_text = full_text or description
                    date_field = result['r'].get('open_date', '')
                else:  # recall
                    subject = result['r'].get('subject', '')
                    consequence = result['r'].get('consequence', '')
                    full_text = full_text or subject
                    date_field = result['r'].get('recall_date', '')
                
                enriched.append({
                    'id': recall_id,
                    'make': result['make'],
                    'model': result['model'],
                    'year': result['r'].get('year', ''),
                    'component': result['r'].get('component', ''),
                    'components': result['components'],
                    'text': full_text,
                    'date': date_field,
                    'subject': result['r'].get('subject', ''),
                    'summary': result['r'].get('summary', ''),
                    'consequence': result['r'].get('consequence', ''),
                    'score': data['score']
                })
    
    return enriched

print("✅ Función ask() lista")

✅ Función ask() lista


In [38]:
# Test de búsqueda - RECALLS
query = "airbag sensor failure"
results = ask(query, k=5, type_in="recall")

print(f"\n{'='*70}")
print(f"Query: '{query}'")
print(f"Resultados encontrados: {len(results)}")
print(f"{'='*70}\n")

for i, r in enumerate(results[:5], 1):
    print(f"\n{i}. [{r['id']}] {r['make']} {r['model']} ({r['year']}) | Score: {r['score']:.3f}")
    print(f"   Component: {r['component']}")
    print(f"   Date: {r['date']}")
    if r['subject']:
        print(f"   Subject: {r['subject'][:150]}...")
    if r['text']:
        print(f"   Text: {r['text'][:300]}...")
    print(f"   All Components: {', '.join(r['components'][:5])}")

🔍 Encontrados 5 resultados en Qdrant

Query: 'airbag sensor failure'
Resultados encontrados: 5


1. [22V240000] BMW IX (2023) | Score: 0.861
   Component: AIR BAGS: AIR BAG/RESTRAINT CONTROL MODULE:SOFTWARE
   Date: 2022-04-13
   Subject: BMW of North America, LLC (BMW) is recalling certain 2022-2023 iX xDrive40, iX XDrive50, and iX M60 vehicles. The air bag malfunction indicator light ...
   Text: BMW of North America, LLC (BMW) is recalling certain 2022-2023 iX xDrive40, iX XDrive50, and iX M60 vehicles. The air bag malfunction indicator light and display message may not illuminate in the event of a problem with the air bag control or pedestrian protection systems, due to incorrect software....
   All Components: SOFTWARE

2. [21V198000] AUDI A3 (2019) | Score: 0.861
   Component: AIR BAGS:ON-OFF SWITCH ASSEMBLY
   Date: 2021-03-22
   Subject: Volkswagen Group of America, Inc. (Audi) is recalling certain 2015-2020 Audi S3 Sedan, A3 Sedan, 2016-2018 A3 Etron, 2017-2020 RS3 Sedan, and 

## Test de búsqueda - INVESTIGATIONS


In [39]:
# Test de búsqueda - INVESTIGATIONS
query = "airbag inflator rupture"
results = ask(query, k=5, type_in="investigation")

print(f"\n{'='*70}")
print(f"Query: '{query}'")
print(f"Resultados encontrados: {len(results)}")
print(f"{'='*70}\n")

for i, r in enumerate(results[:5], 1):
    print(f"\n{i}. [{r['id']}] {r['make']} {r['model']} ({r['year']}) | Score: {r['score']:.3f}")
    print(f"   Component: {r['component']}")
    print(f"   Date: {r['date']}")
    if r['subject']:
        print(f"   Subject: {r['subject'][:150]}...")
    if r['summary']:
        print(f"   Summary: {r['summary'][:200]}...")
    if r['text']:
        print(f"   Text: {r['text'][:400]}...")
    print(f"   All Components: {', '.join(r['components'][:5])}")


🔍 Encontrados 5 resultados en Qdrant

Query: 'airbag inflator rupture'
Resultados encontrados: 5


1. [EA01014] PONTIAC MONTANA (2000) | Score: 0.926
   Component: AIR BAGS:FRONTAL:DRIVER SIDE:INFLATOR MODULE
   Date: 2001-06-11
   Subject: AIR BAG INFLATOR...
   Text: AIR BAG INFLATOR...
   All Components: INFLATOR MODULE

2. [PE02038] NISSAN XTERRA (2002) | Score: 0.919
   Component: AIR BAGS:FRONTAL
   Date: 2002-03-26
   Subject: DRIVER'S AIR BAG SYSTEM MALFUNCTION...
   Text: DRIVER'S AIR BAG SYSTEM MALFUNCTION...
   All Components: FRONTAL

3. [PE96020] DODGE RAM () | Score: 0.918
   Component: AIR BAGS:FRONTAL:DRIVER SIDE:INFLATOR MODULE
   Date: 1996-03-08
   Subject: AIR BAG INFLATOR FAILURE...
   Summary: THERE IS NO SUMMARY CURRENTLY AVAILABLE...
   Text: AIR BAG INFLATOR FAILURE There is no summary currently available...
   All Components: INFLATOR MODULE

4. [RQ99015] GMC YUKON (1992) | Score: 0.906
   Component: SEATS:FRONT ASSEMBLY:RECLINER
   Date: 1999-07-14
   Subject

c:\Users\moral\anaconda3\Lib\site-packages\neo4j\_sync\work\result.py:625: UserWarning: Expected a result with a single record, but found multiple.
  warn(


## Test de búsqueda - COMPLAINTS


In [40]:
# Test de búsqueda - COMPLAINTS
query = "brake failure while driving"
results = ask(query, k=5, type_in="complaint")

print(f"\n{'='*70}")
print(f"Query: '{query}'")
print(f"Resultados encontrados: {len(results)}")
print(f"{'='*70}\n")

for i, r in enumerate(results[:5], 1):
    print(f"\n{i}. [{r['id']}] {r['make']} {r['model']} ({r['year']}) | Score: {r['score']:.3f}")
    print(f"   Component: {r['component']}")
    print(f"   Date: {r['date']}")
    if r['text']:
        print(f"   Description: {r['text'][:500]}...")
    print(f"   All Components: {', '.join(r['components'][:5])}")


🔍 Encontrados 5 resultados en Qdrant

Query: 'brake failure while driving'
Resultados encontrados: 5


1. [2105193] FORD BRONCO SPORT (2021) | Score: 0.955
   Component: SERVICE BRAKES
   Date: 2025-06-30
   Description: Brake Valve Failure while driving...
   All Components: SERVICE BRAKES

2. [1921743] CHRYSLER SEBRING (2008) | Score: 0.944
   Component: SERVICE BRAKES
   Date: 2023-08-23
   Description: Brakes stopped working while in drive...
   All Components: SERVICE BRAKES

3. [1888560] RAM 2500 (2018) | Score: 0.943
   Component: SERVICE BRAKES
   Date: 2023-04-18
   Description: abs brake failure...
   All Components: SERVICE BRAKES

4. [533689] GMC SIERRA (2000) | Score: 0.943
   Component: SERVICE BRAKES, HYDRAULIC:ANTILOCK/TRACTION CONTROL/ELECTRONIC LIMITED SLIP
   Date: 2005-05-05
   Description: BRAKE FAILURE...
   All Components: SERVICE BRAKES, HYDRAULIC:ANTILOCK/TRACTION CONTROL/ELECTRONIC LIMITED SLIP

5. [23945] OLDSMOBILE CUTLASS (1984) | Score: 0.940
   Component: